# Mi vista previa de fechas de servicio

Consulté las fuentes Silver y esta derivación de lectura mediante Genie el 11 de noviembre de 2026. Conservo las tres consultas verificadas y el [informe](../../../docs/service_dates_preview.md); no guardo outputs ni filas fuente en Git. El contenedor del notebook todavía no tiene una ejecución completa documentada como cuaderno.

**Horizonte acordado para la primera implementación: 2026-01-01 a 2026-12-31 del snapshot 2026-09-29.** La comparación de 2025 que aparece en la tercera consulta es solo un escenario de diagnóstico, no otro elemento del Job ni un calendario festivo verificado de 2025.

Una fila de service_dates representa un servicio activo en una fecha, dentro del mismo snapshot. No representa un viaje ni un pasajero. Estas celdas solo leen: no crean tablas ni forman parte del Job.

## Bloque 1 — Fuentes Silver y tipos

Recuperé 29 campos físicos entre calendar y calendar_dates. Compruebo tipos date/int y las columnas técnicas de cada entidad; este resultado no es el esquema de la futura tabla derivada.

In [ ]:
%sql
-- BLOQUE 1 — Vista previa de lectura verificada mediante Genie.
select table_name,column_name,ordinal_position,data_type from workspace.information_schema.columns where table_schema='gtfs_bogota' and table_name in ('silver_gtfs_calendar','silver_gtfs_calendar_dates') order by table_name,ordinal_position;


## Bloque 2 — Conteos y versiones de reglas

Los cuatro grupos confirman 7 / 7 patrones y 140 / 133 excepciones. Las claves son únicas en esos conteos y los marcadores calendar_v1/calendar_dates_v1 son los esperados. Estos sí son conteos consultados en Silver, distintos de los del perfil Bronze anterior.

In [ ]:
%sql
-- BLOQUE 2 — Vista previa de lectura verificada mediante Genie.
select 'calendar' as entity,source_feed_version,source_sha256,count(*) as rows,count(distinct service_id) as distinct_keys,min(start_date) as first_service_date,max(end_date) as last_service_date,cast(null as bigint) as added_rules,cast(null as bigint) as removed_rules,sum(case when not(silver_rule_version <=> 'calendar_v1') then 1 else 0 end) as wrong_rule_version_rows
from workspace.gtfs_bogota.silver_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) group by source_feed_version,source_sha256
union all
select 'calendar_dates',source_feed_version,source_sha256,count(*),count(distinct named_struct('service_id',service_id,'date',`date`)),min(`date`),max(`date`),sum(case when exception_type=1 then 1 else 0 end),sum(case when exception_type=2 then 1 else 0 end),sum(case when not(silver_rule_version <=> 'calendar_dates_v1') then 1 else 0 end)
from workspace.gtfs_bogota.silver_gtfs_calendar_dates where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) group by source_feed_version,source_sha256
order by entity,source_feed_version;


## Bloque 3 — Escenarios de días activos

El recorrido recorta al año analizado, expande el patrón semanal, une las adiciones y elimina las retiradas. Para 2026 devuelve 1.403 pares servicio-fecha; para 2025 devuelve 1.670 sin ninguna excepción de ese año. La selección de 2026 evita interpretar este último número como un calendario festivo completo. Los 19 / 114 avisos son reglas: una adición ya activa no duplica y una retirada ya inactiva no resta de nuevo.

In [ ]:
%sql
-- BLOQUE 3 — Vista previa de lectura verificada mediante Genie.
with c as (select service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,source_feed_version,source_sha256 from workspace.gtfs_bogota.silver_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))),
e as (select service_id,`date` as service_date,exception_type,source_feed_version,source_sha256 from workspace.gtfs_bogota.silver_gtfs_calendar_dates where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))),
bounds as (select distinct source_feed_version,source_sha256,make_date(year(cast(source_feed_version as date)),1,1) as window_start,make_date(year(cast(source_feed_version as date)),12,31) as window_end from c),
expanded as (select c.*,explode(case when greatest(c.start_date,b.window_start)<=least(c.end_date,b.window_end) then sequence(greatest(c.start_date,b.window_start),least(c.end_date,b.window_end),interval 1 day) else cast(array() as array<date>) end) as service_date from c join bounds b using(source_feed_version,source_sha256)),
weekly as (select distinct source_feed_version,source_sha256,service_id,service_date from expanded where element_at(array(monday,tuesday,wednesday,thursday,friday,saturday,sunday),pmod(dayofweek(service_date)+5,7)+1)=1),
in_window as (select e.* from e join bounds b using(source_feed_version,source_sha256) where e.service_date between b.window_start and b.window_end),
additions as (select source_feed_version,source_sha256,service_id,service_date from in_window where exception_type=1),
removals as (select source_feed_version,source_sha256,service_id,service_date from in_window where exception_type=2),
union_days as (select * from weekly union select * from additions),
active as (select u.* from union_days u left anti join removals r on u.source_feed_version=r.source_feed_version and u.source_sha256=r.source_sha256 and u.service_id=r.service_id and u.service_date=r.service_date),
weekly_counts as (select source_feed_version,source_sha256,count(*) as weekly_days from weekly group by source_feed_version,source_sha256),
rule_counts as (select source_feed_version,source_sha256,count(*) as rules_in_window,sum(case when exception_type=1 then 1 else 0 end) as added_rules_in_window,sum(case when exception_type=2 then 1 else 0 end) as removed_rules_in_window from in_window group by source_feed_version,source_sha256),
active_counts as (select source_feed_version,source_sha256,count(*) as active_service_days,count(distinct service_id) as active_services,min(service_date) as first_active_date,max(service_date) as last_active_date from active group by source_feed_version,source_sha256)
select b.source_feed_version,b.source_sha256,b.window_start,b.window_end,coalesce(w.weekly_days,0) as weekly_days,coalesce(r.rules_in_window,0) as rules_in_window,coalesce(r.added_rules_in_window,0) as added_rules_in_window,coalesce(r.removed_rules_in_window,0) as removed_rules_in_window,coalesce(a.active_service_days,0) as active_service_days,coalesce(a.active_services,0) as active_services,a.first_active_date,a.last_active_date
from bounds b left join weekly_counts w using(source_feed_version,source_sha256) left join rule_counts r using(source_feed_version,source_sha256) left join active_counts a using(source_feed_version,source_sha256) order by b.source_feed_version;


## Cómo continúa la implementación

El [contrato derivado](../../../docs/contracts/service_dates.md) requiere horizonte y versiones Delta explícitos. El procesador no simula un TXT nuevo, no modifica las fuentes y no utiliza esta ventana para decidir si existe un service_id.

La validación/publicación pertenece al notebook 10 y al pipeline versionado. No añado este cuaderno al Job. La primera publicación de service_dates de 2026 ya está comprobada: 1.403 pares únicos y siete servicios; el informe enlazado registra el cierre.